# 09 â€” Optuna Tuning: Classification
Each Optuna trial = one hyperparameter combo scored by 5-fold CV (MAE for regression / PR-AUC for classification). Fold-level scores feed `MedianPruner`; XGBoost/LightGBM use early stopping on an inner validation slice of each training fold. SVM models are tuned on a fixed 5000-row subsample for tractability.

In [1]:

import sys, os
from pathlib import Path
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [2]:
import json
import pandas as pd
from src.config import (SPLITS_DIR, RESULTS_DIR, OPTUNA_DIR, FEATURES,
                        TARGET_CLF, N_TRIALS_DEFAULT, N_TRIALS_SVM)
from src.tune import tune_model, make_est
from src.evaluate import evaluate_target, leaderboard_row
from src.models import CLF_NAMES

train = pd.read_csv(SPLITS_DIR / "train.csv")
X = train[FEATURES]
y = train[TARGET_CLF].astype(int)
spw_full = float((y == 0).sum()) / max((y == 1).sum(), 1)
N_TRIALS = {"svc": N_TRIALS_SVM}
rows = []
for name in CLF_NAMES:
    n_trials = N_TRIALS.get(name, N_TRIALS_DEFAULT)
    print(f"tuning {name} | trials={n_trials}", flush=True)
    bf_path = OPTUNA_DIR / f"clf_{name}_{TARGET_CLF}_best.json"
    if bf_path.exists():
        best = json.load(open(bf_path))
        print("  resuming saved best:", round(best["best_value"], 4), flush=True)
    else:
        study, best = tune_model(
            "clf", name, X, y, n_trials, study_name=f"{name}_{TARGET_CLF}")
    q = dict(best["best_params"])
    scores = evaluate_target(
        "clf", lambda q=q, name=name: make_est("clf", name, q, spw=spw_full), X, y)
    row = leaderboard_row(f"{name}_tuned", TARGET_CLF, "clf", scores, extra={
        "optuna_best_value": best["best_value"],
        "best_params": json.dumps(best["best_params"])})
    rows.append(row)
    print("  best CV", best["metric"], round(best["best_value"], 4),
          "| re-scored pr_auc", row["pr_auc_mean"], flush=True)
pd.DataFrame(rows).to_csv(RESULTS_DIR / f"clf_tuned_cv_{TARGET_CLF}.csv", index=False)
print("done")


tuning logistic_regression | trials=50


  resuming saved best: 0.7769


  best CV pr_auc 0.7769 | re-scored pr_auc 0.77692


tuning random_forest | trials=50


  resuming saved best: 0.8777


...

  resuming saved best: 0.8343


  best CV pr_auc 0.8343 | re-scored pr_auc 0.83361


tuning knn | trials=50


  best CV pr_auc 0.8682 | re-scored pr_auc 0.86822


done


## Added diagnostics — Optuna convergence & hyperparameter importances\nLoaded from the persisted study databases (no re-tuning). Optimization history shows whether the search converged; importances show which hyperparameters mattered.

In [2]:
PREFIX = "clf"
for db in sorted(glob.glob(str(OPTUNA_DIR / f"{PREFIX}_*.db"))):
    storage = f"sqlite:///{db}"
    for sname in optuna.get_all_study_names(storage):
        try:
            study = optuna.load_study(study_name=sname, storage=storage)
            n_done = len(study.get_trials(states=(TrialState.COMPLETE,)))
            if n_done < 2:
                print(f"SKIPPING {sname}: only {n_done} completed trials")
                continue
            print(f"{sname}: {n_done} complete trials | best {study.best_value:.4f}", flush=True)
            hist = plot_optimization_history(study)
            hist.update_layout(title=f"{sname} - optimization history", height=360)
        except Exception as e:
            print(f"SKIPPING {sname}: {e}")
            continue
        try:
            imp = plot_param_importances(study)
            imp.update_layout(title=f"{sname} - hyperparameter importances", height=360)
            imp.write_html(FIGURES_DIR / f"optuna_{sname}_importances.html")
        except Exception as e:
            imp = None
            print("  importance plot skipped:", e)
        hist.write_html(FIGURES_DIR / f"optuna_{sname}_history.html")
        display(hist)
        if imp is not None:
            display(imp)


knn_next_is_irregular: 28 complete trials | best 0.8682


lightgbm_next_is_irregular: 32 complete trials | best 0.8691


SKIPPING clf_logistic_regression_next_is_irregular: only 0 completed trials
logistic_regression_next_is_irregular: 100 complete trials | best 0.7769


random_forest_next_is_irregular: 29 complete trials | best 0.8777


svc_next_is_irregular: 11 complete trials | best 0.8343


xgboost_next_is_irregular: 32 complete trials | best 0.8683
